<a href="https://colab.research.google.com/github/Reybolouri/Assignment_3_fashion-mnist/blob/master/forcasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas statsmodels openpyxl


In [3]:
import pandas as pd

url = "https://raw.githubusercontent.com/Reybolouri/Semester-Project-ECON8310-Reybolouri/main/data/ECEC%202022%20Student%20Sign%20In%20and%20Out.xlsx"
df2022 = pd.read_excel(url, engine="openpyxl", header=[0,1], skiprows=5)
# …etc.



In [5]:

!pip install --quiet pandas numpy statsmodels openpyxl



In [6]:

import pandas as pd
import numpy as np
import re
from datetime import datetime, timedelta
from statsmodels.tsa.statespace.sarimax import SARIMAX

# 1) Loader + cleaner for one ECEC file
def load_and_clean_ecec_file(file_path, year):
    # Read with two header rows and skip first 5 metadata lines
    df_raw = pd.read_excel(
        file_path, header=[0,1], skiprows=5, engine="openpyxl"
    )
    # Split metadata vs. timestamp columns
    meta_df       = df_raw.iloc[:, :7].copy()
    timestamps_df = df_raw.iloc[:, 7:].copy()
    # Flatten column names
    meta_df.columns = [
        '_'.join([str(c) for c in col if 'Unnamed' not in str(c)]).strip()
        for col in meta_df.columns
    ]
    timestamps_df.columns = [
        f"{str(c[0]).strip()}_{str(c[1]).strip()}"
        for c in timestamps_df.columns
    ]
    # Add row_id for merging
    meta_df['row_id']       = meta_df.index
    timestamps_df['row_id'] = timestamps_df.index
    # Melt wide → long
    long_df = timestamps_df.melt(
        id_vars='row_id', var_name='date_inout', value_name='timestamp'
    )
    long_df[['date_str','in_out']] = long_df['date_inout']\
        .str.extract(r'(.*)_(IN|OUT)', expand=True)
    # Parse date (month/day + year)
    long_df['date_str'] = long_df['date_str'].str.strip() + f" {year}"
    long_df['date']     = pd.to_datetime(
        long_df['date_str'], format='%b %d %Y', errors='coerce'
    )
    # Pivot so that each row_id+date has one IN and one OUT
    pivot_df = long_df.pivot_table(
        index=['row_id','date'], columns='in_out',
        values='timestamp', aggfunc='first'
    ).reset_index()
    # Ensure both columns exist
    for c in ['IN','OUT']:
        if c not in pivot_df.columns:
            pivot_df[c] = pd.NaT
    # Merge back metadata
    final_df = pivot_df.merge(meta_df, on='row_id', how='left')
    # Clean raw time strings
    def clean_time_only(x):
        if pd.isna(x): return x
        m = re.match(r'^\s*\d{1,2}:\d{2}\s*(AM|PM)', str(x), re.IGNORECASE)
        return m.group(0) if m else x
    final_df['IN']  = final_df['IN'].apply(clean_time_only)
    final_df['OUT'] = final_df['OUT'].apply(clean_time_only)
    final_df['year'] = int(year)
    # Return core columns
    return final_df[['Record ID','Student Status','Room','Tags','date','IN','OUT','year']]

# 2) Function to expand sessions into 30-min blocks and compute staffing
def build_staffing_grid(all_years_df):
    # Extract age_group
    pattern = r'(Infants|Multi-Age|Toddlers|Preschool|Pre-K)'
    all_years_df['age_group'] = all_years_df['Room'].str.extract(pattern, expand=False)
    # Treat “--” as missing OUT
    all_years_df['OUT'] = all_years_df['OUT'].replace('--', pd.NA)
    # Build full datetime fields
    all_years_df['in_datetime'] = pd.to_datetime(
        all_years_df['date'].dt.strftime('%Y-%m-%d') + ' ' + all_years_df['IN'],
        format='%Y-%m-%d %I:%M %p', errors='coerce'
    )
    all_years_df['out_datetime'] = pd.to_datetime(
        all_years_df['date'].dt.strftime('%Y-%m-%d') + ' ' + all_years_df['OUT'],
        format='%Y-%m-%d %I:%M %p', errors='coerce'
    )
    # Drop incomplete sessions
    attended = all_years_df.dropna(subset=['in_datetime','out_datetime']).copy()
    # Generate 30-min blocks
    def blocks(s,e):
        return pd.date_range(start=s, end=e, freq='30min').tolist()
    attended['time_blocks'] = attended.apply(
        lambda r: blocks(r['in_datetime'],r['out_datetime']), axis=1
    )
    expl = attended.explode('time_blocks')
    expl['time_block'] = expl['time_blocks'].dt.floor('30min')
    grid = expl[['Record ID','Student Status','age_group','time_block']].copy()
    # Count children & apply ratios
    grp = grid.groupby(['age_group','time_block','Student Status'])\
              .agg(children_present=('Record ID','nunique'))\
              .reset_index()
    ratios = pd.DataFrame({
        'age_group':['Infants','Multi-Age','Toddlers','Preschool','Pre-K'],
        'student_to_staff':[4,4,6,10,12]
    })
    grp = grp.merge(ratios,on='age_group',how='left')
    grp['staff_required'] = np.ceil(grp['children_present']/grp['student_to_staff']).astype(int)
    return grp



In [7]:

ecec_files = {
    "2022":"https://raw.githubusercontent.com/Reybolouri/Semester-Project-ECON8310-Reybolouri/main/data/ECEC%202022%20Student%20Sign%20In%20and%20Out.xlsx",
    "2023":"https://raw.githubusercontent.com/Reybolouri/Semester-Project-ECON8310-Reybolouri/main/data/ECEC%202023%20Student%20Sign%20In%20and%20Out.xlsx",
    "2024":"https://raw.githubusercontent.com/Reybolouri/Semester-Project-ECON8310-Reybolouri/main/data/ECEC%202024%20Student%20Sign%20In%20and%20Out.xlsx",
    "2025":"https://raw.githubusercontent.com/Reybolouri/Semester-Project-ECON8310-Reybolouri/main/data/ECEC%202025%2001012025-02282025%20Student%20Sign%20In%20and%20Out.xlsx"
}

# Load + concatenate
all_years = pd.concat(
    [load_and_clean_ecec_file(url, yr) for yr,url in ecec_files.items()],
    ignore_index=True
)
# Quick check
print("Rows after cleaning:", len(all_years))

# Build staffing grid
staffing = build_staffing_grid(all_years)
print("Rows in 30-min staffing grid:", staffing.shape[0])

# Save intermediate
staffing.to_csv("ecec_staffing_grouped.csv", index=False)


Rows after cleaning: 61472
Rows in 30-min staffing grid: 122413


In [10]:
#Typical Week & Next-Week Forecast + Accuracy
# === Cell 5: Forecast using Prophet ===
!pip install --quiet prophet

import pandas as pd
from prophet import Prophet

# 1) Load your staffing grid
df = pd.read_csv("ecec_staffing_grouped.csv", parse_dates=["time_block"])

# 2) Build a time series of total staff required
ts = (
    df[df["Student Status"]=="Active"]
      .groupby("time_block")["staff_required"]
      .sum()
      .sort_index()
)

# 3) Prepare for Prophet: ds (datetime) & y (value)
df_prophet = ts.reset_index().rename(columns={"time_block":"ds","staff_required":"y"})

# 4) Fit Prophet with daily + weekly seasonality
m = Prophet(daily_seasonality=True, weekly_seasonality=True, yearly_seasonality=False)
m.fit(df_prophet)

# 5) Make future frame: include history + next 7 days of 30-min steps
horizon = 7*48
future = m.make_future_dataframe(periods=horizon, freq='30min', include_history=True)

# 6) Forecast
forecast = m.predict(future)

# --- Typical Week: use seasonal components only ---
forecast["seasonal"] = forecast["daily"] + forecast["weekly"]
# Grab one full week’s seasonal curve (e.g. starting Mon 2025-01-06)
start = pd.Timestamp("2025-01-06")
end   = start + pd.Timedelta(days=7) - pd.Timedelta(minutes=30)
typical_week = forecast.set_index("ds").loc[start:end, "seasonal"]
typical_week.to_csv("typical_week_prophet.csv", header=["staff_required"])

# --- Next Week: point & interval forecasts ---
next_week = forecast.set_index("ds").iloc[-horizon:]
next_week[["yhat","yhat_lower","yhat_upper"]].to_csv("next_week_prophet.csv")

# 7) Quick peek
print("Typical week (first 5 slots):")
print(typical_week.head(), "\n")
print("Next week forecast (first 5 rows):")
print(next_week[["yhat","yhat_lower","yhat_upper"]].head())




DEBUG:cmdstanpy:input tempfile: /tmp/tmpmb3c4zh8/vgb8ssws.json
DEBUG:cmdstanpy:input tempfile: /tmp/tmpmb3c4zh8/3s0g9gi7.json
DEBUG:cmdstanpy:idx 0
DEBUG:cmdstanpy:running CmdStan, num_threads: None
DEBUG:cmdstanpy:CmdStan args: ['/usr/local/lib/python3.11/dist-packages/prophet/stan_model/prophet_model.bin', 'random', 'seed=69166', 'data', 'file=/tmp/tmpmb3c4zh8/vgb8ssws.json', 'init=/tmp/tmpmb3c4zh8/3s0g9gi7.json', 'output', 'file=/tmp/tmpmb3c4zh8/prophet_model10074mzy/prophet_model-20250503032736.csv', 'method=optimize', 'algorithm=lbfgs', 'iter=10000']
03:27:36 - cmdstanpy - INFO - Chain [1] start processing
INFO:cmdstanpy:Chain [1] start processing
03:27:38 - cmdstanpy - INFO - Chain [1] done processing
INFO:cmdstanpy:Chain [1] done processing


Typical week (first 5 slots):
ds
2025-01-06 06:30:00    1.292098
2025-01-06 07:00:00    4.327762
2025-01-06 07:30:00    6.480328
2025-01-06 08:00:00    7.895364
2025-01-06 08:30:00    8.768215
Name: seasonal, dtype: float64 

Next week forecast (first 5 rows):
                         yhat  yhat_lower  yhat_upper
ds                                                   
2025-02-28 18:00:00  5.300619    3.352716    7.118297
2025-02-28 18:30:00  3.654917    1.697613    5.567825
2025-02-28 19:00:00  2.755520    0.917109    4.737855
2025-02-28 19:30:00  2.830539    0.943068    4.765928
2025-02-28 20:00:00  3.921415    1.880688    5.680501




---



In [ ]:
# #Typical Week & Next-Week Forecast + Accuracy
# # Load the grid
# df = pd.read_csv("ecec_staffing_grouped.csv", parse_dates=["time_block"])
# # Build time series of total staff needed
# ts = (
#     df[df["Student Status"]=="Active"]
#     .groupby("time_block")["staff_required"]
#     .sum()
#     .sort_index()
# )

# # Split train/test
# h = 7*48
# train = ts.iloc[:-h]
# test  = ts.iloc[-h:]

# # 1) SARIMAX next-week
# model = SARIMAX(train,
#                 order=(1,0,1),
#                 seasonal_order=(1,1,1,336),
#                 enforce_stationarity=False,
#                 enforce_invertibility=False)
# res = model.fit(disp=False)
# fc_sarima = res.get_forecast(steps=h).predicted_mean

# # 2) Typical-week average
# df_ts = ts.to_frame("staff")
# df_ts["dow"]=df_ts.index.dayofweek
# df_ts["hr"]=df_ts.index.hour
# df_ts["mn"]=df_ts.index.minute
# typ = df_ts.groupby(["dow","hr","mn"])["staff"].mean()
# fc_typ = test.index.to_series().apply(
#     lambda dt: typ.loc[(dt.dayofweek, dt.hour, dt.minute)]
# )

# # Accuracy metrics
# import numpy as np
# def metrics(f, t):
#     mae  = np.mean(np.abs(f-t))
#     rmse = np.sqrt(np.mean((f-t)**2))
#     mape = np.mean(np.abs((f-t)/t))*100
#     return mae, rmse, mape

# mae_s, rmse_s, mape_s = metrics(fc_sarima, test)
# mae_t, rmse_t, mape_t = metrics(fc_typ,   test)

# print("\nAccuracy on hold-out week:")
# print(f"SARIMAX    → MAE={mae_s:.2f}, RMSE={rmse_s:.2f}, MAPE={mape_s:.1f}%")
# print(f"Typical Wk → MAE={mae_t:.2f}, RMSE={rmse_t:.2f}, MAPE={mape_t:.1f}%")

# # Save forecasts
# pd.Series(fc_sarima, name="SARIMAX").to_csv("next_week_sarima.csv")
# pd.Series(fc_typ,   name="Typical").to_csv("next_week_typical.csv")
# pd.Series(ts[-336:], name="Actual").to_csv("actual_week.csv")

In [15]:
# XGBoost Regression
!pip install --quiet xgboost scikit-learn

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1) Load the staffing grid
df = pd.read_csv("ecec_staffing_grouped.csv", parse_dates=["time_block"])

# 2) Build the target series: total staff required per block
ts = (
    df[df["Student Status"]=="Active"]
      .groupby("time_block")["staff_required"]
      .sum()
      .sort_index()
)

# 3) Create a features DataFrame
data = ts.reset_index().rename(columns={"time_block":"ds","staff_required":"y"})
data["dow"]    = data["ds"].dt.dayofweek
data["hour"]   = data["ds"].dt.hour
data["minute"] = data["ds"].dt.minute
# Cyclical encoding for time features
data["dow_sin"]    = np.sin(2*np.pi*data["dow"]/7)
data["dow_cos"]    = np.cos(2*np.pi*data["dow"]/7)
data["hour_sin"]   = np.sin(2*np.pi*data["hour"]/24)
data["hour_cos"]   = np.cos(2*np.pi*data["hour"]/24)
data["min_sin"]    = np.sin(2*np.pi*data["minute"]/60)
data["min_cos"]    = np.cos(2*np.pi*data["minute"]/60)

# 4) Train/test split: last week as test
h = 7 * 48
train_df = data.iloc[:-h]
test_df  = data.iloc[-h:]

X_train = train_df[["dow_sin","dow_cos","hour_sin","hour_cos","min_sin","min_cos"]]
y_train = train_df["y"]
X_test  = test_df[ ["dow_sin","dow_cos","hour_sin","hour_cos","min_sin","min_cos"]]
y_test  = test_df["y"]

# 5) Fit XGBoost regressor
model = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    objective="reg:squarederror",
    random_state=42
)
model.fit(X_train, y_train)

# 6) Predict & compute accuracy
y_pred = model.predict(X_test)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mape = np.mean(np.abs((y_pred - y_test) / y_test)) * 100

print("XGBoost hold-out accuracy:")
print(f"MAE   = {mae:.2f}")
print(f"RMSE  = {rmse:.2f}")
print(f"MAPE% = {mape:.1f}%")

# 7) Save the next-week predictions vs actual
pd.Series(y_pred, index=test_df["ds"]).to_csv("next_week_xgb_pred.csv", header=["y_pred"])
pd.Series(y_test.values, index=test_df["ds"]).to_csv("actual_week.csv",   header=["y_actual"])



XGBoost hold-out accuracy:
MAE   = 5.99
RMSE  = 6.69
MAPE% = 46.9%


In [1]:

!pip install --quiet pandas numpy statsmodels matplotlib

In [ ]:

import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
import matplotlib.pyplot as plt

# Load the staffing grid
df = pd.read_csv('ecec_staffing_grouped.csv', parse_dates=['time_block'])

# Aggregate to a single time series of total staff required
ts = (
    df[df['Student Status']=='Active']
      .groupby('time_block')['staff_required']
      .sum()
      .sort_index()
)

# Ensure a regular 30-minute index by resampling
ts = ts.resample('30T').sum()

# Split into training (all but last week) and test (last week)
horizon = 7 * 48
train = ts.iloc[:-horizon]
test  = ts.iloc[-horizon:]

# Fit a seasonal ARIMA model: ARIMA(1,1,1)(1,1,1)[336]
model = SARIMAX(
    train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 336),
    enforce_stationarity=False,
    enforce_invertibility=False
)
fit = model.fit(disp=False)

# Forecast the next week
forecast = fit.get_forecast(steps=horizon)
mean_fc = forecast.predicted_mean
ci = forecast.conf_int()

# Plot observed vs. forecast
plt.figure()
plt.plot(train.index, train, label='Training')
plt.plot(test.index, test, label='Actual')
plt.plot(mean_fc.index, mean_fc, label='Forecast')
plt.fill_between(ci.index, ci.iloc[:,0], ci.iloc[:,1], alpha=0.3)
plt.title('Seasonal ARIMA Forecast')
plt.xlabel('Time')
plt.ylabel('Staff Required')
plt.legend()
plt.show()


<ipython-input-2-6304d4c2234e>:18: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  ts = ts.resample('30T').sum()
